# MINI PROJECT 07 ONLINE RETAIL II

In [3]:
# IMPORT
import pandas as pd
import numpy as np

In [4]:
# LOAD DATASET
df = pd.read_excel("online_retail_II.xlsx")
df.shape

(525461, 8)

In [5]:
# BASIC INSPECTION
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [6]:
df.info

<bound method DataFrame.info of        Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
525456  538171     22271                 FELTCRAFT DOLL ROSIE         2   
525457  538171     22750         FELTCRAFT PRINCESS LOLA DOLL         1   
525458  538171     22751       FELTCRAFT PRINCESS OLIVIA DOLL         1   
525459  538171     20970   PINK FLORAL FELTCRAFT SHOULDER BAG         2   
525460  538171     21931               JUMBO STORAGE BAG SUKI         2   

               InvoiceDate  Price  Customer ID         Country  
0 

In [8]:
# MISSING VALUES
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [11]:
# DUPLICATE ROWS
df.duplicated().sum()

6865

In [13]:
# COLUMN NAMES
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='object')

In [15]:
# CHECK DATATYPES
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

### Data Cleaning

In [19]:
df = df.drop_duplicates()
df.shape

(518596, 8)

In [20]:
# Check missing description
df[df["Description"].isnull()].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [21]:
# We'll remove rows where the product description is missing because product-level analysis requires a valid product description.
df = df.dropna(subset=["Description"])
df.shape

(515668, 8)

In [22]:
# Verify Cleaning
df.duplicated().sum()

0

In [23]:
df.isnull().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    104905
Country             0
dtype: int64

In [25]:
# Core Business Matrix
df["Sales"] = df["Quantity"] * df["Price"]
df[[
     "Quantity" , "Price" , "Sales"
]].head(10)

,Quantity,Price,Sales
0,12,6.95,83.4
1,12,6.75,81.0
2,12,6.75,81.0
3,48,2.10,100.8
4,24,1.25,30.0
5,24,1.65,39.6
6,24,1.25,30.0
7,10,5.95,59.5
8,12,2.55,30.6
9,12,3.75,45.0


### CUSTOMER ANALYSIS

In [27]:
customer_df = df.dropna(subset=["Customer ID"]).copy()

In [30]:
# customer KPI
customer_analysis = customer_df.groupby("Customer ID").agg(
     Total_Sales = ("Sales" , "sum"),
     Total_Orders = ("Invoice" , "nunique"),
     Total_Quantity = ("Quantity" , "sum")
)
customer_analysis.head()

,Total_Sales,Total_Orders,Total_Quantity
Customer ID,,,
12346.0,-51.74,15,53
12347.0,1323.32,2,828
12348.0,222.16,1,373
12349.0,2646.99,4,988
12351.0,300.93,1,261


In [32]:
# customer contribution
total_Sales = customer_analysis["Total_Sales"].sum()

customer_analysis["Sales_Contribution_%"] = (
     customer_analysis["Total_Sales"]
     .div(total_Sales).mul(100)
)

In [33]:
top_customers = customer_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
).head(10)
top_customers

,Total_Sales,Total_Orders,Total_Quantity,Sales_Contribution_%
Customer ID,,,,
18102.0,341776.73,95,122988,4.110435
14646.0,243853.05,87,168501,2.932739
14156.0,183163.55,138,106883,2.202847
14911.0,137710.76,270,66572,1.656201
13694.0,128172.42,105,123639,1.541487
17511.0,83760.60,42,54901,1.007361
15061.0,82162.74,90,51395,0.988144
16684.0,75610.17,34,51705,0.909338
13089.0,55801.73,132,28422,0.671109


### PRODUCT ANALYSIS

In [34]:
product_analysis = df.groupby(
    ["StockCode", "Description"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Quantity=("Quantity", "sum"),
    Total_Invoices=("Invoice", "nunique")
)

product_analysis.head()

Total_Sales  Total_Quantity  \
StockCode Description                                                
10002     INFLATABLE POLITICAL GLOBE       5450.21            6748   
10080     GROOVY CACTUS INFLATABLE           10.20              12   
10109     BENDY COLOUR PENCILS                1.68               4   
10120     DOGGY RUBBER                       99.27             457   
          Zebra invcing error                 0.00           -9000   

                                       Total_Invoices  
StockCode Description                                  
10002     INFLATABLE POLITICAL GLOBE              317  
10080     GROOVY CACTUS INFLATABLE                  5  
10109     BENDY COLOUR PENCILS                      1  
10120     DOGGY RUBBER                             48  
          Zebra invcing error                       1

In [ ]:
# Top 10 products by revenue
top_products = product_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
)
top_products.head(10)

,,Total_Sales,Total_Quantity,Total_Invoices
StockCode,Description,,,
22423,REGENCY CAKESTAND 3 TIER,162885.71,13080,2186
85123A,WHITE HANGING HEART T-LIGHT HOLDER,155540.37,57327,3375
DOT,DOTCOM POSTAGE,116401.99,731,735
84879,ASSORTED COLOUR BIRD ORNAMENT,72251.32,44805,1420
22086,PAPER CHAIN KIT 50'S CHRISTMAS,57808.25,17062,964
85099B,JUMBO BAG RED RETROSPOT,54217.92,30249,1253
47566,PARTY BUNTING,49626.92,10084,1017
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47641.89,13663,351
POST,POSTAGE,46092.36,2154,862


In [38]:
# Product contribution
product_analysis["Sales_Contribution_%"] = (
    product_analysis["Total_Sales"]
    .div(product_analysis["Total_Sales"].sum())
    .mul(100)
)

In [39]:
top_products = product_analysis.sort_values(
     "Total_Sales" ,
     ascending = False
)
top_products.head(10)

,,Total_Sales,Total_Quantity,Total_Invoices,Sales_Contribution_%
StockCode,Description,,,,
22423,REGENCY CAKESTAND 3 TIER,162885.71,13080,2186,1.713545
85123A,WHITE HANGING HEART T-LIGHT HOLDER,155540.37,57327,3375,1.636272
DOT,DOTCOM POSTAGE,116401.99,731,735,1.224540
84879,ASSORTED COLOUR BIRD ORNAMENT,72251.32,44805,1420,0.760078
22086,PAPER CHAIN KIT 50'S CHRISTMAS,57808.25,17062,964,0.608138
85099B,JUMBO BAG RED RETROSPOT,54217.92,30249,1253,0.570368
47566,PARTY BUNTING,49626.92,10084,1017,0.522071
84347,ROTATING SILVER ANGELS T-LIGHT HLDR,47641.89,13663,351,0.501189
POST,POSTAGE,46092.36,2154,862,0.484888
